# NB03 — Operating point, discrimination & calibration *(light, CPU)*

Reads the predictions from NB02 (`results/predictions/*.npz`) and returns, **per model** (mean ± SD in
5 folds + pooled 95% bootstrap CI):
- **Discrimination:** AUROC, AUPRC.
- **Clinical operation:** sensitivity @ target-specificity (90/95%), specificity @ target-sensitivity
  (90/95/99%), and **Youden's** threshold (with F1, balanced-acc, MCC at that point).
- **Calibration:** ECE, MCE, Brier + reliability diagram.

Generates EN/PT figures (ROC, PR, calibration, comparison) and consolidates them in
`results/metrics/consolidated.json`. **Does not use GPU.**


In [ ]:
import sys, os, json
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd()/'configs').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
from configs import config as C
from src import metrics as M, plots, utils
import numpy as np, pandas as pd
from sklearn.metrics import roc_curve, precision_recall_curve

print('models:', list(C.MODELS.keys()))
print('folds:', C.N_FOLDS, '| pred dir:', C.PRED_DIR)
print('preds found:', len(list(C.PRED_DIR.glob('*_preds.npz'))), '/ expected', len(C.MODELS)*C.N_FOLDS)


## 1. Load predictions and pool by model
For each model, we gather predictions from the 5 test folds (each image appears in
exactly 1 test fold → the *pool* covers the whole dataset once).

In [ ]:
def load_model_preds(model_key):
    ys, ps, names, fold_ids = [], [], [], []
    for f in range(C.N_FOLDS):
        fp = C.PRED_DIR / f'{model_key}_fold{f}_preds.npz'
        if not fp.exists():
            print(f'  WARNING: missing {fp.name}'); continue
        z = np.load(fp, allow_pickle=True)
        ys.append(z['y']); ps.append(z['p']); names.append(z['image_name'])
        fold_ids.append(np.full(len(z['y']), f))
    return (np.concatenate(ys), np.concatenate(ps),
            np.concatenate(names), np.concatenate(fold_ids))

PRED = {m: load_model_preds(m) for m in C.MODELS}
for m,(y,p,n,fi) in PRED.items():
    print(f'{m:<16} pooled n={len(y)} | prev={y.mean():.3f} | p[{p.min():.3f},{p.max():.3f}]')


## 2. Discrimination: *pooled* estimate (95% bootstrap CI) + stability across folds
Two **distinct** estimators, so as not to confuse them:
- **`*_pool` + 95% CI:** AUROC/AUPRC calculated in the *pool* (all predictions together, each image 1×),
  with 95% bootstrap CI in the same *pool* — the point estimate and the interval come from the **same** base (consistent).
- **`*_fold_mean ± sd`:** mean and standard deviation among the 5 folds — measures **stability** (not the basis for the CI).


In [ ]:
def per_fold_disc(model_key):
    au, ap = [], []
    for f in range(C.N_FOLDS):
        fp = C.PRED_DIR / f'{model_key}_fold{f}_preds.npz'
        if not fp.exists(): continue
        z = np.load(fp, allow_pickle=True)
        d = M.discrimination(z['y'], z['p']); au.append(d['auroc']); ap.append(d['auprc'])
    return np.array(au), np.array(ap)

rows = []
for m in C.MODELS:
    y, p, _, _ = PRED[m]                       # pool
    pooled = M.discrimination(y, p)            # pooled point
    ci_auroc = M.bootstrap_ci(y, p, lambda yy,pp: M.discrimination(yy,pp)['auroc'], C.BOOTSTRAP_N)
    ci_auprc = M.bootstrap_ci(y, p, lambda yy,pp: M.discrimination(yy,pp)['auprc'], C.BOOTSTRAP_N)
    au, ap = per_fold_disc(m)                   # stability across folds
    rows.append({'model': m,
                 'auroc_pool': pooled['auroc'], 'auroc_lo': ci_auroc['lo'], 'auroc_hi': ci_auroc['hi'],
                 'auprc_pool': pooled['auprc'], 'auprc_lo': ci_auprc['lo'], 'auprc_hi': ci_auprc['hi'],
                 'auroc_fold_mean': au.mean(), 'auroc_fold_sd': au.std(),
                 'auprc_fold_mean': ap.mean(), 'auprc_fold_sd': ap.std()})
disc_df = pd.DataFrame(rows).sort_values('auprc_pool', ascending=False).reset_index(drop=True)
display(disc_df.round(4))


## 3. Clinical operating points (on the *pool*)
- **Youden:** threshold that maximizes sens+spec−1 (statistical reference).
- **Triage gate:** we want **high sensitivity** → we report the achievable specificity
  at target-sens 90/95/99% (how much is filtered without missing sick patients) and sensitivity at target-spec 90/95%.

In [ ]:
def unbiased_operating_points(y_pool, p_pool, fold_ids):
    # Cross-validation thresholding to avoid data leakage
    y_pred_bin = {'youden': np.zeros_like(y_pool)}
    for s in C.TARGET_SENS: y_pred_bin[f'sens>={int(s*100)}'] = np.zeros_like(y_pool)
    for sp in C.TARGET_SPEC: y_pred_bin[f'spec>={int(sp*100)}'] = np.zeros_like(y_pool)
    
    thr_means = {k: [] for k in y_pred_bin.keys()}
    for f in range(C.N_FOLDS):
        mask_train = (fold_ids != f)
        mask_test  = (fold_ids == f)
        y_tr, p_tr = y_pool[mask_train], p_pool[mask_train]
        p_te = p_pool[mask_test]
        
        t_j = M.youden_threshold(y_tr, p_tr)
        y_pred_bin['youden'][mask_test] = (p_te >= t_j)
        thr_means['youden'].append(t_j)
        
        for s in C.TARGET_SENS:
            t_s = M.threshold_for_sensitivity(y_tr, p_tr, s)
            k = f'sens>={int(s*100)}'
            y_pred_bin[k][mask_test] = (p_te >= t_s)
            thr_means[k].append(t_s)
            
        for sp in C.TARGET_SPEC:
            t_sp = M.threshold_for_specificity(y_tr, p_tr, sp)
            k = f'spec>={int(sp*100)}'
            y_pred_bin[k][mask_test] = (p_te >= t_sp)
            thr_means[k].append(t_sp)

    out = {}
    for k, y_hat in y_pred_bin.items():
        # Uses y_hat directly in at_threshold with p=y_hat and thr=0.5
        metrics = M.at_threshold(y_pool, y_hat, 0.5) 
        metrics['threshold'] = np.mean(thr_means[k])
        out[k] = metrics
    return out

OP = {m: unbiased_operating_points(PRED[m][0], PRED[m][1], PRED[m][3]) for m in C.MODELS}

# lean table of key triage points
op_rows = []
for m in C.MODELS:
    for key in ['youden', 'sens>=95', 'sens>=99', 'spec>=95']:
        d = OP[m][key]
        op_rows.append({'model': m, 'operating_point': key,
                        'threshold': d['threshold'], 'sens': d['sensitivity'],
                        'spec': d['specificity'], 'ppv': d['ppv'], 'npv': d['npv'],
                        'f1': d['f1'], 'bal_acc': d['balanced_acc'], 'mcc': d['mcc']})
op_df = pd.DataFrame(op_rows)
display(op_df.round(3))

## 4. Calibration (ECE / MCE / Brier) per model

In [ ]:
calib = {}
for m in C.MODELS:
    y_pool, p_pool, _, fold_ids = PRED[m]
    eces, mces, briers = [], [], []
    for f in range(C.N_FOLDS):
        mask = (fold_ids == f)
        c_f = M.calibration(y_pool[mask], p_pool[mask], n_bins=C.ECE_BINS)
        eces.append(c_f['ece']); mces.append(c_f['mce']); briers.append(c_f['brier'])
    
    c_pool = M.calibration(y_pool, p_pool, n_bins=C.ECE_BINS)
    calib[m] = {
        'bins': c_pool['bins'],
        'ece': np.mean(eces), 'ece_sd': np.std(eces),
        'mce': np.mean(mces), 'mce_sd': np.std(mces),
        'brier': np.mean(briers), 'brier_sd': np.std(briers)
    }
    print(f'{m:<16} ECE={calib[m]["ece"]:.4f} (±{calib[m]["ece_sd"]:.4f}) | Brier={calib[m]["brier"]:.4f}')

## 5. Figures EN/PT
ROC and PR overlay all models (pool curves); calibration and comparison highlight the best.

In [ ]:
# (a) ROC and PR — all models in the pool
roc_curves, pr_curves = [], []
for m in C.MODELS:
    y, p, _, _ = PRED[m]
    fpr, tpr, _ = roc_curve(y, p)
    prec, rec, _ = precision_recall_curve(y, p)
    d = M.discrimination(y, p)
    roc_curves.append({'model': m, 'fpr': fpr, 'tpr': tpr, 'auc': d['auroc']})
    pr_curves.append({'model': m, 'recall': rec, 'precision': prec, 'auc': d['auprc']})
plots.plot_roc(roc_curves, name='nb03_roc_all')
plots.plot_pr(pr_curves, name='nb03_pr_all')

# (b) calibration of the best model (highest pooled AUPRC)
best = disc_df.iloc[0]['model']
plots.plot_calibration(calib[best]['bins'], calib[best]['ece'], name=f'nb03_calibration_{best}')

# (c) model comparison (pooled AUPRC/AUROC; error bar = SD across folds)
names = list(disc_df['model'])
plots.plot_model_comparison(names, list(disc_df['auprc_fold_mean']), list(disc_df['auprc_fold_sd']),
                            metric_label='AUPRC', name='nb03_model_comparison_auprc')
plots.plot_model_comparison(names, list(disc_df['auroc_fold_mean']), list(disc_df['auroc_fold_sd']),
                            metric_label='AUROC', name='nb03_model_comparison_auroc')
print('best model (AUPRC):', best)
print('figures saved in figures/en and figures/pt (prefix nb03_).')

## 6. Consolidate everything into JSON

In [ ]:
consolidated = {
    'discrimination': disc_df.to_dict(orient='records'),
    'operating_points': {m: OP[m] for m in C.MODELS},
    'calibration': {m: {'ece': calib[m]['ece'], 'mce': calib[m]['mce'],
                        'brier': calib[m]['brier']} for m in C.MODELS},
    'best_model_auprc': best,
    'config': {'target_sens': C.TARGET_SENS, 'target_spec': C.TARGET_SPEC,
               'bootstrap_n': C.BOOTSTRAP_N, 'ece_bins': C.ECE_BINS,
               'n_folds': C.N_FOLDS},
}
utils.save_json(consolidated, C.METRICS_DIR / 'consolidated.json')
op_df.to_csv(C.METRICS_DIR / 'nb03_operating_points.csv', index=False)
disc_df.to_csv(C.METRICS_DIR / 'nb03_discrimination.csv', index=False)
print('saved: results/metrics/consolidated.json (+ CSVs)')
print('Next: NB04 — gate safety (pathology miss-rate) and cascade cost.')